Imports

In [8]:
import re
import unicodedata
import numpy as np
import pandas as pd

## Loading Data

In [9]:
data = pd.read_csv('Data/finnhub_clean.csv')
print('Raw shape:', data.shape)
display(data.head(5))

Raw shape: (5006, 4)


,stock,headline,date,date_only
0,JPM,"Inside ‘Project Eagle,’ JPMorgan’s High-Wire A...",2026-03-28 19:00:00,2026-03-28
1,JPM,"AI Schism Grips Washington as Tech, Labor Vie ...",2026-03-28 13:00:00,2026-03-28
2,JPM,Dip-buyers arrive to pull gold back from brink...,2026-03-28 09:00:52,2026-03-28
3,JPM,How The Standard Chartered (LSE:STAN) Story Is...,2026-03-28 00:15:29,2026-03-28
4,JPM,Epstein victims to get $72.5M from Bank of Ame...,2026-03-27 23:35:53,2026-03-27


## Preprocessing configuration and cleaning

In [10]:
# Choose modeling text strategy: 'keep', 'mask', or 'remove'
ENTITY_MODE = 'mask'

# Full ticker universe for masking
bank_tickers = [
    'JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY',
    'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY',
    'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE'
 ]
BANK_TICKERS = sorted({ticker.upper().strip() for ticker in bank_tickers})

# Optional: map ticker symbols to common aliases to reduce entity-dominated topics
STOCK_ALIAS_MAP = {
    'JPM': [r'jpmorgan', r'j\.p\.\s*morgan'],
    'BAC': [r'bank\s+of\s+america', r'bofa'],
    'WFC': [r'wells\s+fargo'],
    'C': [r'citigroup', r'\bciti\b'],
    'GS': [r'goldman\s+sachs'],
    'MS': [r'morgan\s+stanley'],
    'HSBC': [r'\bhsbc\b'],
    'SAN': [r'santander', r'banco\s+santander'],
    'BCS': [r'barclays'],
    'DB': [r'deutsche\s+bank'],
    'UBS': [r'\bubs\b'],
    'RY': [r'royal\s+bank\s+of\s+canada', r'\brbc\b'],
    'TD': [r'toronto\-dominion', r'toronto\s+dominion'],
    'SCBFF': [r'standard\s+chartered', r'\bstanchart\b'],
    'SCGLY': [r'societe\s+generale', r'société\s+générale', r'\bsocgen\b'],
    'MFG': [r'mizuho\s+financial\s+group', r'\bmizuho\b'],
    'SMFG': [r'sumitomo\s+mitsui', r'\bsmbc\b'],
    'MUFG': [r'mitsubishi\s+ufj', r'\bufj\b'],
    'BK': [r'bank\s+of\s+new\s+york\s+mellon', r'\bbny\b'],
    'STT': [r'state\s+street'],
    'IDCBY': [r'industrial\s+and\s+commercial\s+bank\s+of\s+china', r'\bicbc\b'],
    'ACGBY': [r'agricultural\s+bank\s+of\s+china'],
    'CICHY': [r'china\s+construction\s+bank'],
    'BACHY': [r'bank\s+of\s+china'],
    'BCMXY': [r'bank\s+of\s+communications'],
    'ING': [r'\bing\b\s+groep'],
    'BNPQY': [r'bnp\s+paribas'],
    'GCRLY': [r'credit\s+agricole', r'crédit\s+agricole'],
    'BPCE': [r'\bbpce\b']
}

print('ENTITY_MODE:', ENTITY_MODE)
print('Ticker count configured:', len(BANK_TICKERS))

ENTITY_MODE: mask
Ticker count configured: 29


In [11]:
def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+\|\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+-\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def _mask_ticker_forms(text: str, ticker: str, replacement: str) -> str:
    out = text
    t = re.escape(ticker)
    out = re.sub(rf'\${t}\b', replacement, out)
    out = re.sub(rf'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*{t}\b', replacement, out, flags=re.IGNORECASE)
    out = re.sub(rf'\({t}\)', replacement, out)
    out = re.sub(rf'(?<!_)\b{t}\b(?!_)', replacement, out)
    return out

def handle_stock_entities(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = text
    replacement_ticker = ' __TICKER__ ' if mode == 'mask' else ' '
    replacement_company = ' __COMPANY__ ' if mode == 'mask' else ' '

    if mode in ('mask', 'remove'):
        # Generic cashtags and exchange-prefixed tickers
        cleaned = re.sub(r'\$[A-Z]{1,7}\b', replacement_ticker, cleaned)
        cleaned = re.sub(r'\b(?:NYSE|NASDAQ|AMEX|TSX|LSE|HKEX|TSE)\s*:\s*[A-Z0-9.]{1,10}\b', replacement_ticker, cleaned, flags=re.IGNORECASE)

        # Mask all configured bank tickers globally
        for ticker in BANK_TICKERS:
            cleaned = _mask_ticker_forms(cleaned, ticker, replacement_ticker)

        # Row stock ticker (extra safety)
        row_stock = '' if pd.isna(stock) else str(stock).upper().strip()
        if row_stock:
            cleaned = _mask_ticker_forms(cleaned, row_stock, replacement_ticker)

        # Mask aliases/names
        for ticker, alias_patterns in STOCK_ALIAS_MAP.items():
            for pattern in alias_patterns:
                cleaned = re.sub(pattern, replacement_company, cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

def preprocess_headline(text: str, stock: str, mode: str = 'mask') -> str:
    cleaned = normalize_text(text)
    if mode in ('mask', 'remove', 'keep') and mode != 'keep':
        cleaned = handle_stock_entities(cleaned, stock, mode=mode)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

## Apply preprocessing

In [12]:
required_cols = ['stock', 'headline', 'date']
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = data.copy()
df = df.dropna(subset=['stock', 'headline', 'date']).copy()
df['stock'] = df['stock'].astype(str).str.upper().str.strip()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['date_only'] = df['date'].dt.date

# Preserve raw headline and create variants
df['headline_raw'] = df['headline'].astype(str).str.strip()
df['headline_clean_keep'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='keep'), axis=1)
df['headline_clean_mask'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='mask'), axis=1)
df['headline_clean_remove'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='remove'), axis=1)

if ENTITY_MODE == 'keep':
    df['headline_model'] = df['headline_clean_keep']
elif ENTITY_MODE == 'remove':
    df['headline_model'] = df['headline_clean_remove']
else:
    df['headline_model'] = df['headline_clean_mask']

# Remove empty model headlines and duplicates
df['headline_model'] = df['headline_model'].fillna('').str.strip()
df = df[df['headline_model'].str.len() > 0].copy()
df = df.drop_duplicates(subset=['stock', 'date', 'headline_model']).sort_values('date').reset_index(drop=True)

print('Processed shape:', df.shape)
display(df[['stock', 'date', 'date_only', 'headline_raw', 'headline_model']].head(10))

Processed shape: (5006, 9)


,stock,date,date_only,headline_raw,headline_model
0,ING,2024-09-17 15:00:55,2024-09-17,What are share buybacks?,What are share buybacks?
1,SMFG,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
2,BACHY,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
3,BACHY,2025-04-02 07:33:46,2025-04-02,"Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh..."
4,SMFG,2025-04-02 12:41:01,2025-04-02,Japan's SMBC Plans Stablecoin Development With...,Japan's __COMPANY__ Plans Stablecoin Developme...
5,SMFG,2025-04-02 13:34:13,2025-04-02,Japanese Banking Giant SMBC Explores Stablecoi...,Japanese Banking Giant __COMPANY__ Explores St...
6,MFG,2025-04-03 11:40:29,2025-04-03,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...
7,SMFG,2025-04-03 11:40:29,2025-04-03,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...
8,MFG,2025-04-04 05:30:00,2025-04-04,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...
9,SMFG,2025-04-04 05:30:00,2025-04-04,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...


## Quality checks

In [13]:
summary = {
    'rows': len(df),
    'unique_stocks': int(df['stock'].nunique()),
    'date_min': df['date'].min(),
    'date_max': df['date'].max(),
    'empty_model_headlines': int((df['headline_model'].str.len() == 0).sum())
}
print(summary)

print('\nTop stocks by article count:')
display(df['stock'].value_counts().head(15).to_frame('n_articles'))

df['raw_len'] = df['headline_raw'].str.len()
df['model_len'] = df['headline_model'].str.len()
print('\nHeadline length stats (raw vs model):')
display(df[['raw_len', 'model_len']].describe().T)

print('\nExamples of transformations:')
display(df[['stock', 'headline_raw', 'headline_clean_mask', 'headline_clean_remove', 'headline_model']].head(10))

{'rows': 5006, 'unique_stocks': 27, 'date_min': Timestamp('2024-09-17 15:00:55'), 'date_max': Timestamp('2026-03-28 19:00:00'), 'empty_model_headlines': 0}

Top stocks by article count:


,n_articles
stock,
BCS,248
DB,248
MUFG,248
JPM,247
GS,246
SAN,243
HSBC,242
MS,242
RY,241



Headline length stats (raw vs model):


,count,mean,std,min,25%,50%,75%,max
raw_len,5006.0,76.849780,35.077129,10.0,59.0,70.0,86.0,485.0
model_len,5006.0,77.873951,35.373807,10.0,60.0,71.0,87.0,487.0



Examples of transformations:


,stock,headline_raw,headline_clean_mask,headline_clean_remove,headline_model
0,ING,What are share buybacks?,What are share buybacks?,What are share buybacks?,What are share buybacks?
1,SMFG,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
2,BACHY,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
3,BACHY,"Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh..."
4,SMFG,Japan's SMBC Plans Stablecoin Development With...,Japan's __COMPANY__ Plans Stablecoin Developme...,Japan's Plans Stablecoin Development With Ava ...,Japan's __COMPANY__ Plans Stablecoin Developme...
5,SMFG,Japanese Banking Giant SMBC Explores Stablecoi...,Japanese Banking Giant __COMPANY__ Explores St...,Japanese Banking Giant Explores Stablecoin Use...,Japanese Banking Giant __COMPANY__ Explores St...
6,MFG,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...
7,SMFG,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...,Shares of banking companies are trading lower ...
8,MFG,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...
9,SMFG,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...,Japan Megabank Stocks Plunge As Trump Tariffs ...


In [14]:
# Coverage check: any configured bank tickers still visible in headline_model?
remaining_counts = {}
for ticker in BANK_TICKERS:
    pattern = rf'(?<!_)\b{re.escape(ticker)}\b(?!_)'
    count = int(df['headline_model'].astype(str).str.contains(pattern, regex=True).sum())
    if count > 0:
        remaining_counts[ticker] = count

print('\nRemaining visible configured tickers in headline_model:', len(remaining_counts))
if remaining_counts:
    display(pd.DataFrame(sorted(remaining_counts.items(), key=lambda x: x[1], reverse=True), columns=['ticker', 'count']).head(20))
else:
    print('✅ All configured bank tickers are masked/removed from headline_model.')


Remaining visible configured tickers in headline_model: 0
✅ All configured bank tickers are masked/removed from headline_model.


## Save outputs for downstream pipelines

In [15]:
output_main = 'Data/finnhub_preprocessed.csv'
output_model = 'Data/finnhub_model_input.csv'

df.to_csv(output_main, index=False)

model_df = df[['stock', 'date', 'date_only', 'headline_model', 'headline_raw']].copy()
model_df = model_df.rename(columns={'headline_model': 'headline'})
model_df.to_csv(output_model, index=False)

print('Saved:')
print(f'  {output_main} -> {df.shape}')
print(f'  {output_model} -> {model_df.shape}')

Saved:
  Data/finnhub_preprocessed.csv -> (5006, 11)
  Data/finnhub_model_input.csv -> (5006, 5)
